# Identifying stage-specific miRNA signatures via Deep Learning

### Deep learning methods
- **Feed-Forward Neural Network (FNN)**
- **Denoising Autoencoder (DAE)**
- **1D Convolutional Neural Network (CNN1D)**
- **Long Short-Term Memory (LSTMSeq)**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import Sequential, Model, Input
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten, LSTM, Reshape
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1_l2

In [2]:
SEEDS = [41, 42, 43, 44, 45]
label_col = "Label"
stage_codes = [1, 2, 3, 4]
stage_map = {1: "Stage_I", 2: "Stage_II", 3: "Stage_III", 4: "Stage_IV"}
k_values = [5, 10, 15]
out_dir = Path("TeamsExports/DeepLearning")

for method in ["FNN", "DAE", "CNN1D", "LSTMSeq"]:
    (out_dir / method).mkdir(parents=True, exist_ok=True)

In [3]:
def build_fnn(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(16, activation='relu', kernel_regularizer=l1_l2(l1=0.001, l2=0.001)),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_dae(input_dim):
    encoder_input = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(encoder_input)
    encoded = Dropout(0.3)(encoded)
    bottleneck = Dense(16, activation='relu', kernel_regularizer=l1_l2(l1=0.001, l2=0.001))(encoded)
    decoded = Dense(64, activation='relu')(bottleneck)
    decoded = Dropout(0.3)(decoded)
    decoder_output = Dense(input_dim, activation='linear')(decoded)
    
    autoencoder = Model(encoder_input, decoder_output)
    autoencoder.compile(optimizer='adam', loss='mse')
    
    classifier = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(16, activation='relu', kernel_regularizer=l1_l2(l1=0.001, l2=0.001)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(input_dim, activation='linear'),
        Dense(1, activation='sigmoid')
    ])
    classifier.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                      loss='binary_crossentropy', metrics=['accuracy'])
    return classifier

def build_cnn1d(input_dim):
    model = Sequential([
        Input(shape=(input_dim, 1)),
        Conv1D(16, kernel_size=3, activation='relu', kernel_regularizer=l1_l2(l1=0.001, l2=0.001)),
        Dropout(0.3),
        Flatten(),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_lstm(input_dim):
    model = Sequential([
        Input(shape=(input_dim, 1)),
        LSTM(16, kernel_regularizer=l1_l2(l1=0.001, l2=0.001)),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

def get_importance(model, method_name, X_train):
    if method_name in ["FNN", "DAE"]:
        dense_layers = [l for l in model.layers if isinstance(l, Dense)]
        W = dense_layers[0].get_weights()[0]
        return np.abs(W).sum(axis=1)
    else:
        X_tensor = tf.constant(X_train, dtype=tf.float32)
        X_tensor = tf.reshape(X_tensor, (-1, X_tensor.shape[1], 1))
        with tf.GradientTape() as tape:
            tape.watch(X_tensor)
            predictions = model(X_tensor, training=False)
        gradients = tape.gradient(predictions, X_tensor)
        importance = tf.reduce_mean(tf.abs(gradients), axis=(0, 2)).numpy()
        return importance

In [4]:
df = pd.read_csv("breast_cancer.csv")

methods = {
    "FNN": build_fnn,
    "DAE": build_dae,
    "CNN1D": build_cnn1d,
    "LSTMSeq": build_lstm
}

for stage_code in stage_codes:
    stage_slug = stage_map[stage_code]
    
    df_stage = df[df[label_col].isin([0, stage_code])].copy()
    X = df_stage.drop(columns=[label_col])
    y = df_stage[label_col].replace({0: 0, stage_code: 1})
    feature_names = X.columns
    
    counts = y.value_counts().to_dict()
    minority = min(counts.values())
    test_size = 0.20 if minority < 30 else 0.30
    val_size = 0.10 if minority < 30 else 0.20
    
    can_stratify = all(v >= 2 for v in counts.values())
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y if can_stratify else None, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size/(1-test_size),
        stratify=y_temp if can_stratify else None, random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    classes = np.array([0, 1])
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight = {0: float(cw[0]), 1: float(cw[1])}
    
    for method_name, build_func in methods.items():
        importances = []
        
        for s in SEEDS:
            tf.random.set_seed(s)
            np.random.seed(s)
            
            model = build_func(X_train_scaled.shape[1])
            
            if method_name in ["CNN1D", "LSTMSeq"]:
                X_tr = X_train_scaled.reshape(-1, X_train_scaled.shape[1], 1)
                X_v = X_val_scaled.reshape(-1, X_val_scaled.shape[1], 1)
            else:
                X_tr, X_v = X_train_scaled, X_val_scaled
            
            es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
            
            model.fit(
                X_tr, y_train,
                epochs=100,
                batch_size=32,
                validation_data=(X_v, y_val),
                callbacks=[es],
                verbose=0,
                class_weight=class_weight
            )
            
            imp = get_importance(model, method_name, X_train_scaled)
            importances.append(imp)
        
        importance = np.mean(importances, axis=0)
        feature_importance = pd.Series(importance, index=feature_names).sort_values(ascending=False)
        
        for k in k_values:
            top_k = feature_importance.head(k).index.tolist()
            pd.Series(top_k, name="miRNA").to_csv(
                out_dir / method_name / f"{method_name}_top{k}_{stage_slug}.csv",
                index=False
            )

print("Deep learning feature selection complete. Files saved to TeamsExports/DeepLearning/")

Deep learning feature selection complete. Files saved to TeamsExports/DeepLearning/
